***
***

# **Counting the ~~Uncounted~~**: 
# *Mapping the GBV Reporting Gap across Kenya*
***
***




**Author:** [Madlene Oloo]  
**Date:** May 1 2026  
**Tools:** Python · requests · BeautifulSoup · matplotlib · openpyxl  

## The Problem

> Every year in Kenya, thousands of women are assaulted, tell no one in authority 
> and thus, are never counted, 
> and yet, we are making policy decisions based on the women we did count.

Gender-based violence is not just a crisis of harm. It is a crisis of invisibility.
When a survivor does not report to police because of fear, distance, stigma, or 
simply because there is no station nearby, she disappears from the official record.
Funding goes where the numbers point. Services are built where cases are filed.
The women who need help most are the ones least likely to appear in the data 
that determines whether help arrives.

This project asks one question:

> **Which counties in Kenya have the largest gap between how many women 
> experience gender-based violence and how many cases are officially recorded?**

The answer tells us not where violence is worst but where the system is 
most blind.

***

## Why This, Why Now

The Kenya Demographic and Health Survey 2022 (the most comprehensive 
county-level survey of women's experiences in Kenya) was released in 2023.
It documents county-by-county data on GBV prevalence 
from over 32,000 households across all 47 counties.

That data has never been placed side by side with Kenya Police Service crime 
reports in a visual, accessible format that a program officer, a county gender 
desk officer, or a policy maker could actually use.

This project does that.

***

## The Research Approach

This analysis follows the same logic as a research paper by triangulating 
across three sources of increasing specificity:

**1. The World Bank Gender Data API** establishes Kenya's national baseline.
This is the globally recognized, internationally standardised picture of where 
Kenya sits on GBV indicators over time. It answers: what do we already know?

**2. The Kenya Demographic and Health Survey (KDHS) 2022** is the primary 
field evidence. Conducted by KNBS across all 47 counties, this is the most 
authoritative Kenya-specific dataset available. It answers: what did women 
actually report experiencing, county by county?

**3. The Kenya Police Service Annual Crime Report** is the counterargument.
This is what the state officially recorded. It answers: what did the system 
actually see?

The gap between source 2 and source 3 is the finding.

***

## What This Project Produces

- A **gap ratio** for each county: how many women reported experiencing violence 
  for every one case that reached the police
- A **ranked bar chart** of all counties by their underreporting ratio
- An **Excel report** that any NGO or county government office could open, 
  filter, and act on tomorrow

---

## Who Needs This

A program officer at an NGO deciding where to open a safe house.  
A county gender desk officer making the case for more resources.  
A donor allocating funding across regions.  

All of them are currently working with incomplete information.  
This project is a step toward closing that gap.

## *Section 1:* Kenya Gender Data from World Bank API

In [11]:
# Establisng the national baseline
# Pull data from the World Bank's Gender Data Portal
# These Gender indicators show physical violence, 
# sexual violence & attitudes toward violence.

import requests
import json

# Create dictionary to store the Indicators & Indicators ID
indicators = {
    "Physical Violence (% of women ages 15-49)": "WB.GS.SG.VAW.1549.ZS",
    "Sexual Violence (% of women ages 15-49)": "SG.VAW.AFSX.ZS",
    "Husband is Justified to Beat Wife (% agree)": "SG.VAW.REAS.ZS"
              }

indicators.keys() 
indicators.values()
indicators.items()
indicator_name = "indicators.keys()"
indicator_id = "indicators.values()"

# Country Code: KEN
# Source: 14 (Gender Data Portal)
# Requests will start with the base_url and 
# add specific code indicator at the end.

BASE_url = "https://api.worldbank.org/v2/country/KE/indicator"

# Fetch fuction to avoid writing logic 3 times
      
def fetch_indicator(indicator_name, indicator_id):
    # Build full URL
    # Combine base_url + code + JSON format + upto 20 years of data
    url = f"{BASE_url}/{indicator_id}?format=json&per_page=20"
    
    print(f"\nFetching: {indicator_name}")
    print(f"URL: {url}")
    
    # Make the request
    response = requests.get(url) # Make request

    # Check response
    if response.status_code == 200:
        print(f"Request Successful. Status code: {response.status_code}")
    
    else:
        print(f"Request Failed. Status code: {response.status_code}")

    # Read and transform text to list 
    data = response.json() # Convert text into object
    
    if len(data) < 2 or data[1] is None:
        return []
    # Loop through years returned
    # For data found, build dict {indicator_name, year,
    # value(rounded to 2 decimal places & country name)}.
    records = []
    for entry in data[1]:
        if entry["value"] is not None:
            records.append({
                "indicator": indicator_name,
                "year": int(entry["date"]),
                "value": round(entry["value"], 2),
                "country": entry["country"]["value"]
            })
    print(f"Found {len(records)} years of data")        
    return records

# Call the function *3 
national_data = {}
for name, id in indicators.items():
    national_data[name] = fetch_indicator(name, id)
print("\nKENYA NATIONAL BASELINE — SUMMARY")
    # Sort records by newest year first
for indicator_name, records in national_data.items():
    if records:
        # Sort by year descending so the most recent appears first
        records_sorted = sorted(records, key=lambda x: x["year"], reverse=True)
        most_recent = records_sorted[0]
        print(f"\n{indicator_name}")
        print(f"  Most recent data point: {most_recent['year']}")
        print(f"  Value: {most_recent['value']}%")
        print(f"  All years available: {[r['year'] for r in records_sorted]}")
    else:
        print(f"\n{indicator_name}")
        print(f"  No data available")


Fetching: Physical Violence (% of women ages 15-49)
URL: https://api.worldbank.org/v2/country/KE/indicator/WB.GS.SG.VAW.1549.ZS?format=json&per_page=20
Request Successful. Status code: 200

Fetching: Sexual Violence (% of women ages 15-49)
URL: https://api.worldbank.org/v2/country/KE/indicator/SG.VAW.AFSX.ZS?format=json&per_page=20
Request Successful. Status code: 200
Found 2 years of data

Fetching: Husband is Justified to Beat Wife (% agree)
URL: https://api.worldbank.org/v2/country/KE/indicator/SG.VAW.REAS.ZS?format=json&per_page=20
Request Successful. Status code: 200
Found 3 years of data

KENYA NATIONAL BASELINE — SUMMARY

Physical Violence (% of women ages 15-49)
  No data available

Sexual Violence (% of women ages 15-49)
  Most recent data point: 2014
  Value: 14.1%
  All years available: [2014, 2009]

Husband is Justified to Beat Wife (% agree)
  Most recent data point: 2022
  Value: 32.4%
  All years available: [2022, 2014, 2009]


> When looking for the data on Physical violence:
> The World Bank API returned no results for Kenya 
> on the physical violence indicator (WB.GS.SG.1549.ZS) for 2022.

- The World Bank publishes this indicator based on national Demographic and Health Surveys. Kenya's most recent DHS was conducted in 2022, but the World Bank has not yet processed those results into their API. 
- The most recent data point available through the API is from 2014.

- This gap between when data is collected and when it becomes accessible to researchers and policymakers is itself evidence of the problem this project is investigating: the systems designed to track GBV are slow, incomplete, and often several years behind reality.

***

## *Section 2, Part 1:* Kenya Demographic and Health Survey (KDHS) 2022

> This data has been dowloaded and loaded directly from the Kenya 
> Demographic and Health Survey 2022 that feeds the national 
> data captured by WorldBank gender data.

- This is the missing data that has yet  to be updated by the 
Worldbank

In [ ]:
# Physical Violence Data
# Loading KDHS 2022 data
# openpyxl reads .xlsx file

import openpyxl 
import re
# Load workbook
FILE_PATH = r"C:\Users\User\Desktop\Python-Capstone-Project\Data\kdhs_2022.xlsx"

print(f"Loading: {FILE_PATH}")

workbook = openpyxl.load_workbook(FILE_PATH, data_only=True)
worksheet = workbook["June 2023"]
matches_found = 0

# Use regex to match 
pattern = re.compile(r'violence|gbv|sexual|abuse|sexual', re.IGNORECASE)

# See all sheet names
print(f"Checking starting from row 3338...\n")

for i, row in enumerate(worksheet.iter_rows(min_row=3338, values_only=True, max_row=3731)):
    row_text = " ".join(str(cell).strip() for cell in row if cell is not None)

    if pattern.search(row_text):
        print(f"MATCH FOUND at Row {i}: {row_text}")
        matches_found += 1

    if i > 3731:
        break

if matches_found == 0:
        print("No matches found. Showing raw rows 3339-3731 for debugging:")
        for row in worksheet.iter_rows(min_row=3338, max_row=3731, values_only=True):
             print(row)
else:
     print(f"\nTotal matches found: {matches_found}")

Loading: C:\Users\User\Desktop\Python-Capstone-Project\Data\kdhs_2022.xlsx
Checking starting from row 3338...
MATCH FOUND at Row 0: 29 Experience of physical violence Percentage  of Women age15 - 49 who experienced physical violence in the last 12 months often 50-54 na
MATCH FOUND at Row 1: 29 Experience of physical violence Percentage  of Women age15 - 49 who experienced physical violence in the last 12 months often Total 15-54 na
MATCH FOUND at Row 2: 29 Experience of physical violence Percentage  of Women age15 - 49 who experienced physical violence in the last 12 months sometimes Age 15-19 10
MATCH FOUND at Row 3: 29 Experience of physical violence Percentage  of Women age15 - 49 who experienced physical violence in the last 12 months sometimes Age 20-24 12.3
MATCH FOUND at Row 4: 29 Experience of physical violence Percentage  of Women age15 - 49 who experienced physical violence in the last 12 months sometimes Age 25-29 14.4
MATCH FOUND at Row 5: 29 Experience of physical violence

***

## *Section 2, Part 2:* Core Data Denoting Geographical Breakdown 

In [ ]:
# Violence Breakdown Per County
# Loading KDHS county crosstab data
# openpyxl reads .xlsx file

import openpyxl
import re
# Load workbook
COUNTIES_FILE_PATH = r"C:\Users\User\Desktop\Python-Capstone-Project\Data\kdhs_counties.xlsx"

print(f"Loading: {COUNTIES_FILE_PATH}")

wb_county = openpyxl.load_workbook(COUNTIES_FILE_PATH, data_only=True)
worksheet = wb_county["Sheet1"]
county_found = 0

# Use regex to match 
header_pattern = re.compile(r'violence|gbv|sexual|abuse|sexual', re.IGNORECASE)

# See all sheet names
print(f"Checking starting from row 8036...")

for i, row in enumerate(worksheet.iter_rows(min_row=8036, values_only=True, max_row=8419)):
    row_text = " ".join(str(cell).strip() for cell in row if cell is not None)

    if header_pattern.search(row_text):
        print(f"Found table header at Row {i}  ")
        found_header = True
        continue

if found_header:
        if any(row):
             print(f"Row {i}: {row}")
             county_found += 1

        elif county_found > 55:
        else:
             break

## Section 3 - Calculate underreporting gap ratio

- Analysis

## Section 4 - Bar chart with matplotlib

- Visualizations

## Section 5 -Export to Excel with openpyxl

- Presentation quality

## Conclusion - Insights & Recommendations

- Accuracy and conclusion